# Boundary Interpolation

Interpolate gaps of low confidence joints at the movement boundaries

### Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

### Config

In [2]:
CROPPED_DIR    = Path('../data/processed/keypoints_cropped')
FULL_DIR       = Path('../data/processed/keypoints')
VIDEO_DIR      = Path('../data/Utvalda filminspelningar för IRAF analys/Dec 2025 sit-stå och stå-sitt')
OUT_CSV_BASE   = Path('../data/processed/keypoints_interpolated')
OUT_VIDEO_BASE = Path('../data/processed/keypoints_interpolated_boundary_videos')

THRESHOLD      = 0.6
MAX_INTERP_GAP = 10
FPS            = 50

MODELS = [
    ('movenet',       'confidence'),
    ('mediapipe_norm','visibility'),
]

JOINTS = [
    'nose',
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

MODEL_COORDS = {
    'movenet': ('x', 'y'),
    'mediapipe_norm': ('x', 'y', 'z'),
}

OUT_CSV_BOUNDARY_BASE = OUT_CSV_BASE.parent / f"{OUT_CSV_BASE.name}_boundary"
OUT_CSV_BOUNDARY_BASE.mkdir(parents=True, exist_ok=True)

OUT_CSV_BOUNDARY_BASE

PosixPath('../data/processed/keypoints_interpolated_boundary')

### Helpers

In [3]:
def good_mask_metric(df, metric_col, threshold):
    v = df[metric_col].to_numpy()
    return (~np.isnan(v)) & (v >= threshold)

def get_frame_to_index(frames):
    return {int(f): i for i, f in enumerate(frames)}

def last_good_before(good_frames, frame):
    b = good_frames[good_frames < frame]
    return int(b[-1]) if len(b) else None

def first_good_after(good_frames, frame):
    a = good_frames[good_frames > frame]
    return int(a[0]) if len(a) else None

def missing_mask_joint(df, joint, coords):
    masks = [df[f'{joint}_{c}'].isna().to_numpy() for c in coords]
    return np.logical_or.reduce(masks)

def leading_missing_end_idx(missing_mask):
    if not missing_mask[0]:
        return None
    good = np.flatnonzero(~missing_mask)
    return (int(good[0]) - 1) if len(good) else (len(missing_mask) - 1)

def trailing_missing_start_idx(missing_mask):
    if not missing_mask[-1]:
        return None
    good = np.flatnonzero(~missing_mask)
    return (int(good[-1]) + 1) if len(good) else 0

def interpolate_frames_between_anchors(df_out, frames_crop, joint, coords,
                                       anchor_before_frame, anchor_after_frame,
                                       full_df, full_frame_to_idx, target_mask):
    i0 = full_frame_to_idx[anchor_before_frame]
    i1 = full_frame_to_idx[anchor_after_frame]

    gap_len = int(anchor_after_frame - anchor_before_frame - 1)
    if gap_len <= 0:
        return

    tgt_frames = frames_crop[target_mask].astype(int)
    k = tgt_frames - anchor_before_frame

    for c in coords:
        col = f'{joint}_{c}'
        v0 = float(full_df.at[i0, col])
        v1 = float(full_df.at[i1, col])
        seq = np.linspace(v0, v1, gap_len + 2)[1:-1]
        df_out.loc[target_mask, col] = seq[k - 1]

### Run

In [4]:
for model, metric in MODELS:
    coords = MODEL_COORDS[model]

    in_interp_dir = OUT_CSV_BASE / model
    in_full_dir   = FULL_DIR / model
    out_dir       = OUT_CSV_BOUNDARY_BASE / model
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_interp_dir.glob('*.csv'))

    for csv_path in files:
        name = csv_path.name

        df_crop = pd.read_csv(csv_path)
        df_full = pd.read_csv(in_full_dir / name)

        frames_crop = df_crop['frame'].to_numpy().astype(int)
        first_frame = int(frames_crop[0])
        last_frame  = int(frames_crop[-1])

        frames_full = df_full['frame'].to_numpy().astype(int)
        full_frame_to_idx = get_frame_to_index(frames_full)

        df_out = df_crop.copy()

        for joint in JOINTS:
            metric_col = f'{joint}_{metric}'
            if metric_col not in df_full.columns:
                continue

            flag_col = f'{joint}_interpolated'
            if flag_col not in df_out.columns:
                df_out[flag_col] = False

            good_frames_full = frames_full[good_mask_metric(df_full, metric_col, THRESHOLD)]

            miss = missing_mask_joint(df_out, joint, coords)

            lead_end = leading_missing_end_idx(miss)
            if lead_end is not None:
                anchor_before = last_good_before(good_frames_full, first_frame)

                after_inside_idx = lead_end + 1
                anchor_after = None
                if after_inside_idx < len(frames_crop):
                    f = int(frames_crop[after_inside_idx])
                    if f in full_frame_to_idx:
                        v = df_full.at[full_frame_to_idx[f], metric_col]
                        if (not pd.isna(v)) and (v >= THRESHOLD):
                            anchor_after = f
                if anchor_after is None:
                    anchor_after = first_good_after(good_frames_full, last_frame)

                if anchor_before is not None and anchor_after is not None:
                    gap = int(anchor_after - anchor_before - 1)
                    if gap <= MAX_INTERP_GAP:
                        mask = (frames_crop >= first_frame) & (frames_crop <= int(frames_crop[lead_end]))
                        interpolate_frames_between_anchors(
                            df_out, frames_crop, joint, coords,
                            anchor_before, anchor_after,
                            df_full, full_frame_to_idx,
                            mask
                        )
                        df_out.loc[mask, flag_col] = True

            miss = missing_mask_joint(df_out, joint, coords)

            trail_start = trailing_missing_start_idx(miss)
            if trail_start is not None:
                anchor_after = first_good_after(good_frames_full, last_frame)

                before_inside_idx = trail_start - 1
                anchor_before = None
                if before_inside_idx >= 0:
                    f = int(frames_crop[before_inside_idx])
                    if f in full_frame_to_idx:
                        v = df_full.at[full_frame_to_idx[f], metric_col]
                        if (not pd.isna(v)) and (v >= THRESHOLD):
                            anchor_before = f
                if anchor_before is None:
                    anchor_before = last_good_before(good_frames_full, first_frame)

                if anchor_before is not None and anchor_after is not None:
                    gap = int(anchor_after - anchor_before - 1)
                    if gap <= MAX_INTERP_GAP:
                        mask = (frames_crop >= int(frames_crop[trail_start])) & (frames_crop <= last_frame)
                        interpolate_frames_between_anchors(
                            df_out, frames_crop, joint, coords,
                            anchor_before, anchor_after,
                            df_full, full_frame_to_idx,
                            mask
                        )
                        df_out.loc[mask, flag_col] = True

        df_out.to_csv(out_dir / name, index=False)

    print(f'{model}: wrote {len(files)} files to {out_dir}')

movenet: wrote 10 files to ../data/processed/keypoints_interpolated_boundary/movenet
mediapipe_norm: wrote 10 files to ../data/processed/keypoints_interpolated_boundary/mediapipe_norm


### Summary

In [5]:
summary = []

for model, _ in MODELS:
    before_dir = OUT_CSV_BASE / model
    after_dir  = OUT_CSV_BOUNDARY_BASE / model

    total_coord_changes = 0
    total_flag_changes  = 0
    total_files         = 0

    for csv_path in sorted(before_dir.glob("*.csv")):
        name = csv_path.name
        df_before = pd.read_csv(csv_path)
        df_after  = pd.read_csv(after_dir / name)

        coord_cols = [c for c in df_before.columns if c.endswith(('_x','_y','_z'))]
        flag_cols  = [c for c in df_before.columns if c.endswith('_interpolated')]

        coord_diff = (df_before[coord_cols].fillna(0) != df_after[coord_cols].fillna(0)).sum().sum()
        flag_diff  = (df_before[flag_cols] != df_after[flag_cols]).sum().sum()

        total_coord_changes += coord_diff
        total_flag_changes  += flag_diff
        total_files += 1

    summary.append({
        "model": model,
        "files": total_files,
        "coord_values_changed": int(total_coord_changes),
        "flags_changed": int(total_flag_changes),
    })

pd.DataFrame(summary)

,model,files,coord_values_changed,flags_changed
0,movenet,10,104,52
1,mediapipe_norm,10,0,0


In [6]:
filled_gaps_all = []

for model, _ in MODELS:
    before_dir = OUT_CSV_BASE / model
    after_dir  = OUT_CSV_BOUNDARY_BASE / model
    coords = MODEL_COORDS[model]

    for csv_path in sorted(before_dir.glob("*.csv")):
        name = csv_path.name
        df_before = pd.read_csv(csv_path)
        df_after  = pd.read_csv(after_dir / name)

        frames = df_before["frame"].to_numpy().astype(int)
        n = len(frames)
        first_frame = int(frames[0])
        last_frame  = int(frames[-1])

        for joint in JOINTS:
            coord_cols = [f"{joint}_{c}" for c in coords if f"{joint}_{c}" in df_before.columns]
            if not coord_cols:
                continue

            was_nan = df_before[coord_cols].isna().any(axis=1).to_numpy()
            now_nan = df_after[coord_cols].isna().any(axis=1).to_numpy()

            filled = was_nan & (~now_nan)
            if not filled.any():
                continue

            padded = np.pad(filled, (1, 1), constant_values=False)
            diff = np.diff(padded.astype(int))
            starts = np.where(diff == 1)[0]
            ends   = np.where(diff == -1)[0]

            for s, e in zip(starts, ends):
                start_idx = int(s)
                end_idx_excl = int(e)
                start_frame = int(frames[start_idx])
                end_frame   = int(frames[end_idx_excl - 1])
                length = int(end_idx_excl - start_idx)

                touches_start = (start_idx == 0)
                touches_end   = (end_idx_excl == n)

                filled_gaps_all.append({
                    "model": model,
                    "file": name,
                    "joint": joint,
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "length_frames": length,
                    "touches_start": touches_start,
                    "touches_end": touches_end,
                    "crop_first_frame": first_frame,
                    "crop_last_frame": last_frame,
                })

filled_gaps_df = pd.DataFrame(filled_gaps_all)
filled_gaps_df

,model,file,joint,start_frame,end_frame,length_frames,touches_start,touches_end,crop_first_frame,crop_last_frame
0,movenet,DJI_20250425092743_0028_D_movenet.csv,left_elbow,495,497,3,True,False,495,588
1,movenet,DJI_20250425093100_0030_D_movenet.csv,left_shoulder,290,290,1,True,False,290,381
2,movenet,DJI_20250425093100_0030_D_movenet.csv,right_elbow,290,290,1,True,False,290,381
3,movenet,DJI_20250425093100_0030_D_movenet.csv,left_wrist,290,290,1,True,False,290,381
4,movenet,DJI_20250425093100_0030_D_movenet.csv,right_wrist,381,381,1,False,True,290,381
5,movenet,DJI_20250425093100_0030_D_movenet.csv,right_hip,290,295,6,True,False,290,381
6,movenet,DJI_20250425093100_0030_D_movenet.csv,right_ankle,290,290,1,True,False,290,381
7,movenet,DJI_20250425104507_0045_D_movenet.csv,nose,398,399,2,False,True,327,399
8,movenet,DJI_20250425104507_0045_D_movenet.csv,left_elbow,327,327,1,True,False,327,399
9,movenet,DJI_20250425104507_0045_D_movenet.csv,right_elbow,398,399,2,False,True,327,399


In [7]:
if len(filled_gaps_df) == 0:
    print("No filled gaps detected.")
else:
    filled_gaps_df["is_boundary_run"] = filled_gaps_df["touches_start"] | filled_gaps_df["touches_end"]

    display(filled_gaps_df["is_boundary_run"].value_counts().to_frame("count"))

    non_boundary = filled_gaps_df[~filled_gaps_df["is_boundary_run"]]
    if len(non_boundary):
        print("WARNING: Found filled runs that do NOT touch a boundary (unexpected).")
        display(non_boundary.sort_values(["model","file","joint","start_frame"]).head(50))
    else:
        print("All filled runs touch a boundary (as expected).")

,count
is_boundary_run,
True,26


All filled runs touch a boundary (as expected).


## Interpolate Z

* if xy is NaN, z should also be NaN
* if xy is interpolated, z should also be interpolated

In [8]:
IN_DIR = OUT_CSV_BOUNDARY_BASE / 'mediapipe_norm'

for csv_path in sorted(IN_DIR.glob('*.csv')):
    df = pd.read_csv(csv_path).reset_index(drop=True)
    n = len(df)

    for joint in JOINTS:
        flag = f'{joint}_interpolated'
        xcol = f'{joint}_x'
        ycol = f'{joint}_y'
        zcol = f'{joint}_z'
        if flag not in df.columns or xcol not in df.columns or ycol not in df.columns or zcol not in df.columns:
            continue

        interp = df[flag].to_numpy().astype(bool)

        padded = np.pad(interp, (1, 1), constant_values=False)
        diff = np.diff(padded.astype(int))
        starts = np.where(diff == 1)[0]
        ends   = np.where(diff == -1)[0]

        for s, e in zip(starts, ends):
            start = int(s)
            end   = int(e)

            if start == 0 or end == n:
                continue

            z0 = df.at[start - 1, zcol]
            z1 = df.at[end, zcol]
            if pd.isna(z0) or pd.isna(z1):
                continue

            gap_len = end - start
            df.loc[start:end - 1, zcol] = np.linspace(z0, z1, gap_len + 2)[1:-1]

        xy_missing = df[xcol].isna() | df[ycol].isna()
        df.loc[xy_missing, zcol] = np.nan

    df.to_csv(csv_path, index=False)

print("MediaPipe z fixed in place (interior runs only):", IN_DIR)

MediaPipe z fixed in place (interior runs only): ../data/processed/keypoints_interpolated_boundary/mediapipe_norm


### Summary

In [9]:
before_dir = OUT_CSV_BASE / "mediapipe_norm"
after_dir  = OUT_CSV_BOUNDARY_BASE / "mediapipe_norm"
rows = []

for p in sorted(before_dir.glob("*.csv")):
    name = p.name
    df_b = pd.read_csv(p)
    df_a = pd.read_csv(after_dir / name)

    frames = df_b["frame"].to_numpy().astype(int)
    n = len(frames)

    for joint in JOINTS:
        flag = f"{joint}_interpolated"
        xcol = f"{joint}_x"
        ycol = f"{joint}_y"
        zcol = f"{joint}_z"
        if flag not in df_a.columns or zcol not in df_b.columns or zcol not in df_a.columns:
            continue
        if xcol not in df_a.columns or ycol not in df_a.columns:
            continue

        z_before = df_b[zcol]
        z_after  = df_a[zcol]

        z_changed = (z_before.fillna(0) != z_after.fillna(0)).to_numpy()
        if not z_changed.any():
            continue

        padded = np.pad(z_changed, (1, 1), constant_values=False)
        diff = np.diff(padded.astype(int))
        starts = np.where(diff == 1)[0]
        ends   = np.where(diff == -1)[0]

        interp = df_a[flag].astype(bool).to_numpy()
        xy_missing = (df_a[xcol].isna() | df_a[ycol].isna()).to_numpy()

        for s, e in zip(starts, ends):
            s = int(s); e = int(e)

            touches_start = (s == 0)
            touches_end   = (e == n)
            all_interp    = bool(interp[s:e].all())
            any_interp    = bool(interp[s:e].any())
            all_xy_missing = bool(xy_missing[s:e].all())

            if all_interp:
                change_type = "z interpolated (xy interpolated)"
            elif all_xy_missing:
                change_type = "z set to NaN (xy missing)"
            elif any_interp:
                change_type = "mixed (some interpolated frames)"
            else:
                change_type = "other (check)"

            rows.append({
                "file": name,
                "joint": joint,
                "start_frame": int(frames[s]),
                "end_frame": int(frames[e - 1]),
                "length": int(e - s),
                "touches_start": touches_start,
                "touches_end": touches_end,
                "change_type": change_type,
            })

analysis_df = pd.DataFrame(rows)

display(
    analysis_df.groupby("change_type")["length"]
    .agg(runs="count", frames="sum", mean_len="mean", max_len="max")
    .sort_values("frames", ascending=False)
)

analysis_df.sort_values(["file", "joint", "start_frame"])

,runs,frames,mean_len,max_len
change_type,,,,
z set to NaN (xy missing),40,1694,42.35,118
z interpolated (xy interpolated),2,4,2.00,3


,file,joint,start_frame,end_frame,length,touches_start,touches_end,change_type
1,DJI_20250425092743_0028_D_mediapipe_norm.csv,left_wrist,507,528,22,False,False,z set to NaN (xy missing)
0,DJI_20250425092743_0028_D_mediapipe_norm.csv,right_elbow,495,588,94,True,True,z set to NaN (xy missing)
6,DJI_20250425092743_0028_D_mediapipe_norm.csv,right_knee,585,588,4,False,True,z set to NaN (xy missing)
2,DJI_20250425092743_0028_D_mediapipe_norm.csv,right_wrist,495,542,48,True,False,z set to NaN (xy missing)
3,DJI_20250425092743_0028_D_mediapipe_norm.csv,right_wrist,548,563,16,False,False,z set to NaN (xy missing)
4,DJI_20250425092743_0028_D_mediapipe_norm.csv,right_wrist,566,568,3,False,False,z interpolated (xy interpolated)
5,DJI_20250425092743_0028_D_mediapipe_norm.csv,right_wrist,575,588,14,False,True,z set to NaN (xy missing)
7,DJI_20250425093100_0030_D_mediapipe_norm.csv,right_elbow,320,364,45,False,False,z set to NaN (xy missing)
8,DJI_20250425093100_0030_D_mediapipe_norm.csv,right_elbow,372,381,10,False,True,z set to NaN (xy missing)
9,DJI_20250425093100_0030_D_mediapipe_norm.csv,right_wrist,329,353,25,False,False,z set to NaN (xy missing)


## Video Output

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio

def draw_keypoints_on_image(image, keypoints):
    height, width, _ = image.shape
    aspect_ratio = float(width) / height
    fig, ax = plt.subplots(figsize=(12 * aspect_ratio, 12))
    fig.tight_layout(pad=0)
    ax.margins(0)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    plt.axis('off')

    ax.imshow(image)

    for joint in JOINTS:
        x = keypoints.get(f"{joint}_x")
        y = keypoints.get(f"{joint}_y")
        if pd.isna(x) or pd.isna(y):
            continue

        interp = bool(keypoints.get(f"{joint}_interpolated", False))
        c = "#00ff00" if not interp else "#ffa500"
        ax.scatter([x * width], [y * height], c=c)

    fig.canvas.draw()
    image_from_plot = np.frombuffer(fig.canvas.tostring_argb(), dtype=np.uint8)
    image_from_plot = image_from_plot.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    image_from_plot = image_from_plot[:, :, 1:4]
    plt.close(fig)

    return image_from_plot


def draw_keypoints_on_video(video_path, keypoints_df, out_path):
    reader = imageio.get_reader(str(video_path))
    fps = reader.get_meta_data().get('fps', FPS)

    start_frame = int(keypoints_df['frame'].iloc[0])
    n_frames = len(keypoints_df)

    with imageio.get_writer(str(out_path), fps=fps) as writer:
        for i in range(n_frames):
            frame = reader.get_data(start_frame + i)
            row = keypoints_df.iloc[i]
            image = draw_keypoints_on_image(frame, row)
            writer.append_data(image)


for model, metric in MODELS:
    out_video = OUT_VIDEO_BASE / model
    out_video.mkdir(parents=True, exist_ok=True)

    for csv_path in sorted((CROPPED_DIR / model).glob('*.csv')):
        match = re.search(r'_0(\d{3})_D_', csv_path.stem)
        if not match:
            continue
        vid_id = match.group(1)

        vid_files = list(VIDEO_DIR.glob(f'*_0{vid_id}_D.MP4'))
        if not vid_files:
            print(f'Video not found: {vid_id}')
            continue

        df_interp = pd.read_csv(OUT_CSV_BOUNDARY_BASE / model / csv_path.name).reset_index(drop=True)

        suffix   = f'_{model}'
        out_name = csv_path.stem.replace(suffix, '') + '_interpolated.mp4'
        out_path = out_video / out_name

        draw_keypoints_on_video(vid_files[0], df_interp, out_path)
        print(f'{model} {vid_id}: {len(df_interp)} frames -> {out_name}')

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


movenet 028: 94 frames -> DJI_20250425092743_0028_D_interpolated.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


movenet 030: 92 frames -> DJI_20250425093100_0030_D_interpolated.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


...

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


movenet 074: 60 frames -> DJI_20250425120835_0074_D_interpolated.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


movenet 076: 151 frames -> DJI_20250425121226_0076_D_interpolated.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
